# 03 — Publish to ArcGIS Online

This notebook publishes the processed site-year GeoDataFrame as a
Hosted Feature Layer on ArcGIS Online.

**Prerequisites:**
- Run `02_pipeline_demo.ipynb` first to generate the processed GeoJSON.
- Set environment variables `AGOL_USERNAME` and `AGOL_PASSWORD`
  (see README.md for details).

In [1]:
import sys
sys.path.insert(0, "..")

import logging
logging.basicConfig(level=logging.INFO, format="%(name)s — %(message)s")

import geopandas as gpd
from pathlib import Path
from src import publish

## Load processed data

In [2]:
SOURCE_ID = "hawaii"
data_path = Path("..") / "data" / "processed" / f"{SOURCE_ID}_site_summary.geojson"

gdf = gpd.read_file(data_path)
print(f"Loaded {len(gdf):,} features from {data_path.name}")
gdf.head(3)

Loaded 1,553 features from hawaii_site_summary.geojson


,site,obs_year,island,region_name,reef_zone,depth_bin,min_depth,max_depth,latitude,longitude,...,std_t3_UI,mean_t3_UNK,std_t3_UNK,mean_t3_UPMA,std_t3_UPMA,mean_t3_USC,std_t3_USC,mean_t3_ZO,std_t3_ZO,geometry
0,FFS-1012,2016,French Frigate,Northwestern Hawaiian Islands,Lagoon,Shallow,NaN,NaN,23.778698,-166.164036,...,0.0,0.33,1.83,0.0,0.0,0.0,0.0,0.0,0.0,POINT (-166.16404 23.7787)
1,FFS-1015,2016,French Frigate,Northwestern Hawaiian Islands,Lagoon,Shallow,NaN,NaN,23.733424,-166.163069,...,0.0,0.97,3.01,0.0,0.0,0.0,0.0,0.0,0.0,POINT (-166.16307 23.73342)
2,FFS-1018,2016,French Frigate,Northwestern Hawaiian Islands,Lagoon,Shallow,NaN,NaN,23.819997,-166.143617,...,0.0,0.33,1.83,0.0,0.0,0.0,0.0,0.0,0.0,POINT (-166.14362 23.82)


## Publish to ArcGIS Online

In [3]:
url = publish.publish_site_summary(
    gdf=gdf,
    source_id=SOURCE_ID,
    mode="create",  # "overwrite" to replace data in place, "append" to add rows
)
print(f"\nPublished Feature Layer URL:\n  {url}")

arcgis.gis._impl._portalpy — Retrieving roles(start=1, num=100)
src.publish — Authenticated to https://www.arcgis.com as calvin.quigley
src.publish — Deleting existing item: CRCP Benthic Cover – Hawaiian Archipelago (8345dc5ea0344568910e890de5a96de1, type=GeoJson)
arcgis.gis._impl._portalpy — Retrieving roles(start=1, num=100)
pyogrio._io — Created 1,553 records
/Users/calquigs/coralnet_explorer/notebooks/../src/publish.py:161: DeprecatedWarning: add is deprecated as of 2.3.0 and has been removed in 3.0.0. Use `Folder.add()` instead.
  item = publish_feature_layer(
/Users/calquigs/coralnet_explorer/venv/lib/python3.14/site-packages/arcgis/graph/data_model_types.py:82: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  class GraphProperty(BaseModel):
/Users/calquigs/coralnet_explorer/venv/lib/python3.14/site-p


Published Feature Layer URL:
  https://services6.arcgis.com/hcrxPNKq5Bune9Ia/arcgis/rest/services/CRCP_Benthic_Cover_Hawaiian_Archipelago/FeatureServer


## Verify: query the published layer

In [4]:
from arcgis.gis import GIS
from arcgis.features import FeatureLayer

fl = FeatureLayer(url + "/0")
count = fl.query(return_count_only=True)
print(f"Feature count on AGOL: {count}")
print(f"Local feature count:   {len(gdf)}")
assert count == len(gdf), "Mismatch between published and local counts!"
print("Counts match.")

Feature count on AGOL: 1553
Local feature count:   1553
Counts match.


## Display inline map

In [6]:
import os
from arcgis.gis import GIS

gis = GIS(
    os.environ.get("AGOL_URL", "https://www.arcgis.com"),
    os.environ.get("AGOL_USERNAME"),
    os.environ.get("AGOL_PASSWORD"),
)

m = gis.map("Hawaii")
m.zoom = 7
m.content.add(FeatureLayer(url + "/0"))
m

arcgis.gis._impl._portalpy — Retrieving roles(start=1, num=100)


Map()